# Fundamentals of Data Science - Case Study 4
## Predicting Student Dropout and Academic Success
- **Student Name:** Alapati Nagesh
- **Roll Number:** 160124733160
- **Department:** Computer Science and Engineering (CSE-3)
- **Dataset:** `Studentdata.csv`

In [ ]:
# Cell 1: Environment Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.sans-serif': 'DejaVu Sans', 'font.size': 11})

In [ ]:
# Cell 2: Part A - Data Ingestion
file_path = 'Studentdata.csv'
df = pd.read_csv(file_path, sep=';')
df.columns = df.columns.str.strip().str.replace('\t', '')

print(f"Observations: {df.shape[0]}, Features: {df.shape[1]}")
print(f"Missing Values: {df.isnull().sum().sum()}")
print(f"Duplicate Records: {df.duplicated().sum()}")
df.head()

In [ ]:
# Cell 3: Part B - Exploratory Data Analysis
target_dist = df['Target'].value_counts()
target_pct = df['Target'].value_counts(normalize=True) * 100
for cls in target_dist.index:
    print(f"{cls}: {target_dist[cls]} ({target_pct[cls]:.2f}%)")

palette = {'Graduate': '#2b7bba', 'Dropout': '#d9534f', 'Enrolled': '#f0ad4e'}

# Figure 1: Target Outcome Distribution
plt.figure(figsize=(7, 4.5))
sns.countplot(data=df, x='Target', order=['Graduate', 'Dropout', 'Enrolled'], palette=palette)
plt.title('Figure 1: Target Class Distribution', fontsize=12, weight='bold')
plt.show()

# Figure 2: Age & Admission Grade Distributions
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.histplot(df['Age at enrollment'], bins=25, kde=True, ax=axes[0], color='#2b5c8f')
axes[0].set_title('Figure 2A: Age at Enrollment Distribution', weight='bold')
sns.histplot(df['Admission grade'], bins=25, kde=True, ax=axes[1], color='#388e3c')
axes[1].set_title('Figure 2B: Admission Grade Distribution', weight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 4: Part B - Bivariate Visualizations
plt.figure(figsize=(8, 4.5))
sns.boxplot(data=df, x='Target', y='Admission grade', order=['Graduate', 'Dropout', 'Enrolled'], palette=palette)
plt.title('Figure 3: Admission Grade Comparison Across Outcome Classes', fontsize=12, weight='bold')
plt.show()

plt.figure(figsize=(8, 4.5))
sns.boxplot(data=df, x='Target', y='Curricular units 1st sem (grade)', order=['Graduate', 'Dropout', 'Enrolled'], palette=palette)
plt.title('Figure 4: 1st Semester Grade Across Outcome Classes', fontsize=12, weight='bold')
plt.show()

In [ ]:
# Cell 5: Part C - Hypothesis Testing
grad_g = df[df['Target'] == 'Graduate']['Curricular units 1st sem (grade)']
drop_g = df[df['Target'] == 'Dropout']['Curricular units 1st sem (grade)']
enr_g = df[df['Target'] == 'Enrolled']['Curricular units 1st sem (grade)']

f_stat, p_val = stats.f_oneway(grad_g, drop_g, enr_g)
kw_stat, kw_pval = stats.kruskal(grad_g, drop_g, enr_g)

print(f"One-Way ANOVA F-Statistic : {f_stat:.4f}, p-value: {p_val:.4e}")
print(f"Kruskal-Wallis H-Statistic: {kw_stat:.4f}, p-value: {kw_pval:.4e}")
if p_val < 0.05:
    print("[RESULT] Reject H0: First-semester grade differences are statistically significant (p < 0.001).")

In [ ]:
# Cell 6: Part D - Multiclass Logistic Regression
X = df.drop(columns=['Target'])
y = df['Target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_reg = LogisticRegression(max_iter=1000, multi_class='multinomial', random_state=42)
log_reg.fit(X_train_scaled, y_train)

y_pred = log_reg.predict(X_test_scaled)

print(f"Overall Accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=4))

plt.figure(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred, labels=['Dropout', 'Enrolled', 'Graduate'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Dropout', 'Enrolled', 'Graduate'], yticklabels=['Dropout', 'Enrolled', 'Graduate'])
plt.title('Figure 5: Confusion Matrix (Multiclass Logistic Regression)', fontsize=12, weight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('Actual True Label')
plt.tight_layout()
plt.show()